In [1]:
import json
import requests

# 方式一：如果通过 API Gateway URL 调用
def call_via_api_gateway():
    url = "https://sptadsyzvugpstxq5t5mfc54yq0aifuh.lambda-url.ap-southeast-1.on.aws/"  # 替换为实际 URL
    
    # 构造业务数据 QueryData
    query_data = {
        "gameType": 450,
        "round": 1,
        "smallblind": 10,
        "szToken": "test_token_123",
        "users": [
            {"nPlayerId": 1001, "seat": 0, "score": 1000, "bRobot": 0}, # 0=真人
            {"nPlayerId": 1002, "seat": 1, "score": 1000, "bRobot": 1}  # 1=机器人
        ]
    }
    
    # 构造完整的请求 Body
    payload = {
        "Cmd": "QueryAISingleData",  # 必须是 QueryAISingleData 或 cacheService
        "QueryData": query_data
    }
    
    # 设置 Header
    # 注意：Lambda 代码 (L521-524) 强制要求 Content-Type 和 Content-Hmac
    headers = {
        "Content-Type": "application/json",
        "Content-Hmac": "dummy_signature"  # 代码仅检查 Key 是否存在，未校验具体签名值
    }
    
    print(f"Sending request to {url}...")
    response = requests.post(url, json=payload, headers=headers)
    print(response.json())
call_via_api_gateway()

Sending request to https://sptadsyzvugpstxq5t5mfc54yq0aifuh.lambda-url.ap-southeast-1.on.aws/...
{'Res': 1, 'ResData': {'bEffect': 1, 'handCards': [{'seat': 1, 'cards': [23, 68], 'nPlayerId': 1002}, {'seat': 0, 'cards': [60, 38], 'nPlayerId': 1001}], 'commonCards': [41, 20, 75, 70, 61], 'bPreset': False, 'inventionType': 0, 'redisScores': {}}, 'Msg': 'Success', 'Delta': 0.020832061767578125}


In [22]:
import json
import requests

# 方式一：如果通过 API Gateway URL 调用
def call_via_api_gateway():
    url = "https://dfrroptihhgz73uwwoecls3wuy0uznfe.lambda-url.ap-southeast-1.on.aws/"  # 替换为实际 URL
    
    # 构造业务数据 QueryData
    query_data = {
        "gameType": 450,
        "round": 1,
        "smallblind": 10,
        "szToken": "test_token_123",
        "users": [
            {"nPlayerId": 1001, "seat": 0, "score": 1000, "bRobot": 0}, # 0=真人
            {"nPlayerId": 1002, "seat": 1, "score": 1000, "bRobot": 1}  # 1=机器人
        ]
    }
    
    # 构造完整的请求 Body
    payload = {
        "Cmd": "QueryAISingleData",  # 必须是 QueryAISingleData 或 cacheService
        "QueryData": query_data
    }
    
    # 设置 Header
    # 注意：Lambda 代码 (L521-524) 强制要求 Content-Type 和 Content-Hmac
    headers = {
        "Content-Type": "application/json",
        "Content-Hmac": "dummy_signature"  # 代码仅检查 Key 是否存在，未校验具体签名值
    }
    
    print(f"Sending request to {url}...")
    response = requests.post(url, json=payload, headers=headers)
    print(response.json())
call_via_api_gateway()

Sending request to https://dfrroptihhgz73uwwoecls3wuy0uznfe.lambda-url.ap-southeast-1.on.aws/...
{'rt': 0.23775529861450195, 'predictions': {'num': 18, 'action': 2}}


In [29]:
# {
#     "my_card": [
#         66,
#         50
#     ],
#     "opponent_card": [
#         59,
#         33
#     ],
#     "opponent_card_1": [
#         28,
#         49
#     ],
#     "board_cards": [
#         58,
#         34,
#         65,
#         60,
#         51
#     ]
# }['my_card']

In [ ]:
import math
import threading
import time
from collections import Counter
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait

import requests

URL = 'https://sptadsyzvugpstxq5t5mfc54yq0aifuh.lambda-url.ap-southeast-1.on.aws/'
HEADERS = {'Content-Type': 'application/json', 'Content-Hmac': 'dummy_signature'}

thread_local = threading.local()

def _get_session():
    s = getattr(thread_local, 'session', None)
    if s is None:
        s = requests.Session()
        adapter = requests.adapters.HTTPAdapter(pool_connections=256, pool_maxsize=256, max_retries=0)
        s.mount('https://', adapter)
        s.mount('http://', adapter)
        thread_local.session = s
    return s

def _build_payload(i: int):
    query_data = {
        'gameType': 450,
        'round': 1,
        'smallblind': 10,
        'szToken': 'test_token_123',
        'users': [
            {'nPlayerId': 1001 + (i % 1000000), 'seat': 0, 'score': 1000, 'bRobot': 0},
            {'nPlayerId': 2001 + (i % 1000000), 'seat': 1, 'score': 1000, 'bRobot': 1},
        ],
    }
    return {'Cmd': 'QueryAISingleData', 'QueryData': query_data}

def _percentile(sorted_values, p):
    if not sorted_values:
        return None
    k = (len(sorted_values) - 1) * (p / 100.0)
    f = math.floor(k)
    c = math.ceil(k)
    if f == c:
        return sorted_values[int(k)]
    d0 = sorted_values[int(f)] * (c - k)
    d1 = sorted_values[int(c)] * (k - f)
    return d0 + d1

def _request_once(i: int, timeout_s: float):
    session = _get_session()
    payload = _build_payload(i)
    t0 = time.perf_counter()
    try:
        r = session.post(URL, json=payload, headers=HEADERS, timeout=timeout_s)
        rt = time.perf_counter() - t0
        ok = (r.status_code == 200)
        err = None
        if ok:
            try:
                data = r.json()
                if isinstance(data, dict) and ('Res' in data):
                    ok = (data.get('Res') == 1)
            except Exception as e:
                ok = False
                err = f'json_error:{type(e).__name__}'
        else:
            err = f'status:{r.status_code}'
        return ok, rt, err, r.status_code
    except Exception as e:
        rt = time.perf_counter() - t0
        return False, rt, f'exception:{type(e).__name__}', None

def run_load_test(total_requests: int = 1000, concurrency: int = 50, timeout_s: float = 3.0, warmup: int = 10):
    for i in range(max(0, warmup)):
        _request_once(i, timeout_s)

    latencies = []
    ok_count = 0
    err_counter = Counter()
    status_counter = Counter()

    started_at = time.perf_counter()
    in_flight = set()
    next_i = 0

    def _submit(ex, i):
        return ex.submit(_request_once, i, timeout_s)

    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        while next_i < total_requests or in_flight:
            while next_i < total_requests and len(in_flight) < concurrency * 2:
                in_flight.add(_submit(ex, next_i))
                next_i += 1

            done, in_flight = wait(in_flight, return_when=FIRST_COMPLETED)
            for fut in done:
                ok, rt, err, status = fut.result()
                latencies.append(rt)
                if ok:
                    ok_count += 1
                else:
                    if err:
                        err_counter[err] += 1
                if status is not None:
                    status_counter[status] += 1

    elapsed = time.perf_counter() - started_at
    latencies_sorted = sorted(latencies)
    total = len(latencies_sorted)
    fail = total - ok_count

    p50 = _percentile(latencies_sorted, 50)
    p90 = _percentile(latencies_sorted, 90)
    p99 = _percentile(latencies_sorted, 99)

    print('URL:', URL)
    print('Total:', total, 'OK:', ok_count, 'Fail:', fail)
    print('Elapsed(s):', round(elapsed, 3), 'RPS:', round(total / elapsed, 2) if elapsed > 0 else None)
    print('Latency(s) p50/p90/p99:', round(p50, 4) if p50 else None, round(p90, 4) if p90 else None, round(p99, 4) if p99 else None)
    print('Status:', dict(status_counter))
    print('Top errors:', err_counter.most_common(5))

run_load_test(total_requests=2000, concurrency=80, timeout_s=3.0, warmup=20)
